# 16_04 — Evaluación predictiva · Escenario 6

Paso **3b** del ciclo. Supone que `16_03_convergencia` dio veredicto ✓; si las
cadenas no convergieron, nada de lo que sigue significa algo.


## Qué se reporta

| Sección | Qué responde |
|---|---|
| 4 | métricas puntuales y distribucionales, **train vs test** |
| 5 | bandas de credibilidad sobre la serie de cada score |
| 6 | intervalos sobre la curva, en extractos cada 10 períodos |
| 7 | ventana móvil: cómo evoluciona el error al cruzar $T_0$ |
| 8 | calibración marginal: PIT por bloque |
| **9** | **cobertura estratificada por el régimen de covarianza** |
| 9.1 | obsolescencia de la base: ¿crece el error de truncamiento tras $t^*$? |
| 10 | comparación con las líneas base |

Todo el cálculo vive en `fit/` y todo el dibujo en `graphics/`: este notebook
sólo orquesta.

## 1. Imports, rutas y artefactos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

from model_psbp_fd.pipelines import (
    cargar_curvas, cargar_curvas_true, cargar_fpca, cargar_estandarizador,
    cargar_datasets_ar, cargar_hiperparametros, cargar_config_evaluacion,
    cargar_escenario,
)
from model_psbp_fd.functions_models import DataStandardizer
from model_psbp_fd.models.pspb_fd_v3 import PSBPPredictor, PropagadorFuncional

from model_psbp_fd.fit import (
    rmse_por_coeficiente, r2_por_columna, razon_dispersion, mise, rmse_funcional,
    crps_muestral, energy_score, cobertura, intervalo_muestral,
    pit_muestral, diagnostico_pit,
    cobertura_condicional,
    agrupar_momentos, ventana_movil_scores, ventana_movil_funcional,
)
from model_psbp_fd.graphics import (
    plot_ventana_movil, plot_bandas_serie, plot_extractos_curvas,
    plot_calibracion_pit, plot_scatter_theta,
)
from model_psbp_fd.utils import get_project_root
from model_psbp_fd.utils.quadrature import pesos_trapezoidales

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

In [ ]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# [CONFIG] debe coincidir con 16_01, 16_03 y psbp_fd_iteracion.m
BASENAME, ESCENARIO_ID, REPLICA_ID = "escenario", 6, 1
EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}"

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID,
    "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID,
}
for r in PATHS.values():
    r.mkdir(parents=True, exist_ok=True)
print(f"EXPERIMENT_ID : {EXPERIMENT_ID}")

In [ ]:
dfs_train, manifest = cargar_datasets_ar(PATHS, bloque="train")
dfs_test,  _        = cargar_datasets_ar(PATHS, bloque="test")
hp_json     = cargar_hiperparametros(PATHS)
eval_config = cargar_config_evaluacion(PATHS)

COMPONENT_IDX = manifest["component_idx"]
n_components  = len(COMPONENT_IDX)
cov_names     = manifest["cov_names"]
N_LAGS        = int(manifest["n_lags"])
T, T0         = int(manifest["T"]), int(manifest["T0"])
N_ITER        = int(hp_json["n_iter"])
MCMC_CFG      = hp_json["mcmc_config"]
BURN          = int(MCMC_CFG["burn"])

assert manifest["scores_scale"] == "standardized_zscore_ddof0"

# Parámetros de evaluación: se LEEN del artefacto, no se redeclaran aquí.
NIVEL        = float(eval_config.get("nivel_credibilidad", 0.95))
MODO_RESIDUO = eval_config.get("modo_residuo", "ninguno")
OBJETIVO     = eval_config.get("objetivo_evaluacion", "curva_verdadera")
VENTANAS_W   = list(eval_config.get("ventana_movil", {}).get("w", [10, 20, 40]))
ESTRAT_CFG   = eval_config.get("estratificacion", {})
REPR_CFG     = eval_config.get("representacion", {})

TQ      = int(REPR_CFG.get("t_quiebre", T0))
N_ADAPT = int(REPR_CFG.get("periodos_adaptacion", 0))

print(f"T={T}  T0={T0}  n_lags={N_LAGS}  componentes={n_components}")
print(f"nivel={NIVEL}  ·  modo_residuo={MODO_RESIDUO!r}  ·  objetivo={OBJETIVO!r}")
print(f"ventanas w = {VENTANAS_W}")
print(f"estratificación = {ESTRAT_CFG.get('variable', '—')} "
      f"({ESTRAT_CFG.get('metodo', '—')})")

print("\n" + "=" * 72)
print("ADVERTENCIA DE LECTURA (declarada en eval_config.json):")
print("\n  [estratificación] " + ESTRAT_CFG.get("advertencia", ""))
print("\n  [Bloque 2] " + REPR_CFG.get("advertencia", ""))
print(f"\n  t* = {TQ}   ·   T0 = {T0}   ·   ¿coinciden? "
      f"{'SÍ' if TQ == T0 else 'NO'}")
print("=" * 72)
ESTRATO_ES_PARTICION = (TQ == T0)

In [ ]:
fpca = cargar_fpca(PATHS)
_ver = fpca.verificar()
assert _ver["todo_ok"], f"Las identidades FPCA no se cumplen: {_ver}"

scores_standardizer = cargar_estandarizador(PATHS, DataStandardizer)
X_obs, grilla = cargar_curvas(PATHS)                 # (T, G) con ruido
X_true        = cargar_curvas_true(PATHS)            # (T, G) verdadera — el objetivo
w_quad        = pesos_trapezoidales(grilla)

M_fpca   = fpca.M
Psi_grid, mu_grid = fpca.Psi_grid, fpca.mu_grid
SCORES     = fpca.SCORES                              # (T, M) escala original
SCORES_STD = scores_standardizer.transform(SCORES)

print(f"FPCA M={M_fpca} K={fpca.K}   ·   curvas {X_true.shape}  grilla {grilla.shape}")
print(f"sd(observada - verdadera) = {(X_obs - X_true).std():.4f}   "
      "← el ruido que el modelo NO debe predecir")
assert COMPONENT_IDX == list(range(M_fpca)), (
    "La propagación funcional necesita el vector completo de scores en orden; "
    f"COMPONENT_IDX={COMPONENT_IDX} y M={M_fpca}.")

### 1.1 Estado verdadero del generador

El régimen de covarianza vigente —pre o post quiebre— es lo que hace evaluable
el eje 2. Se lee del CSV que escribió `16_01`; si faltara, se recupera del `.npz`
crudo comparando cada fila de `interno_trayectoria_espectro` con el espectro
inicial.

Ninguna de estas cantidades entra en predicción alguna: sólo definen la
partición de los orígenes.

In [ ]:
_csv_estado = PATHS["out_report"] / "10_estado_quiebre.csv"
if _csv_estado.exists():
    estado_df = pd.read_csv(_csv_estado)
    quiebre   = estado_df["quiebre_idx"].to_numpy().astype(int)
    en_adapt  = estado_df["en_adaptacion"].to_numpy().astype(bool)
    _fuente   = _csv_estado.name
else:
    _crudo = cargar_escenario(str(PATHS["raw"] / f"escenario_{ESCENARIO_ID}.npz"))
    assert "interno_trayectoria_espectro" in _crudo, (
        "El .npz no contiene `interno_trayectoria_espectro`: regenera 16_01 con "
        "guardar_escenario(..., incluir_internos=True).")
    _lamt = _crudo["interno_trayectoria_espectro"]
    _lam0 = _crudo["interno_espectro_inicial"]
    quiebre  = np.array([0 if np.allclose(_lamt[t], _lam0) else 1
                         for t in range(T)], dtype=int)
    en_adapt = (np.arange(1, T + 1) >= TQ) & (np.arange(1, T + 1) < TQ + N_ADAPT)
    _fuente  = f"escenario_{ESCENARIO_ID}.npz::interno_trayectoria_espectro"

J_ESTRATOS = int(quiebre.max()) + 1
ETIQUETAS  = list(ESTRAT_CFG.get("etiquetas",
                                 ["pre-quiebre", "post-quiebre"]))
assert quiebre.shape == (T,), f"quiebre tiene {quiebre.shape}, se esperaba ({T},)"

print(f"estado verdadero leído de {_fuente}   ·   {J_ESTRATOS} estratos")
print(f"proporciones globales: "
      f"{[round(float((quiebre == j).mean()), 3) for j in range(J_ESTRATOS)]}")
print(f"periodos en adaptación (varianza aún convergiendo): {int(en_adapt.sum())}")

# Verificación numérica contra la fuente: el estrato debe cambiar exactamente
# en t = t*, no uno antes ni uno después.
_primer_post = int(np.argmax(quiebre == 1)) + 1     # base-1
assert _primer_post == TQ, (
    f"El primer periodo post-quiebre es t={_primer_post} y t*={TQ}: revisar la "
    "convención base-1 de t_quiebre.")
print(f"\n  El primer periodo post-quiebre es t={_primer_post} = t*, como debe.")

## 2. Trazas MCMC

In [ ]:
def ruta_traza(fpc_idx, chain):
    return PATHS["out_artefact"] / f"chain_fpc_{fpc_idx}_iter{chain:02d}.mat"


def leer_traza(path):
    m = loadmat(str(path))
    claves = ["betajhout", "beta0hout", "tauhout", "alphahout", "psijhout",
              "Gammajhout", "gammajhout", "pijout", "wjout", "osumout", "inEout"]
    traces = {k: np.asarray(m[k], dtype=np.float64) for k in claves}
    burn = int(np.asarray(m["burn"]).ravel()[0])
    feat = str(np.atleast_1d(m["feature_names"]).ravel()[0]).split(",")
    return traces, burn, feat


class ModeloTraza:
    def __init__(self, traces, burn, feature_names):
        self.traces = traces
        self.feature_names_ = list(feature_names)
        self.burn = int(burn)
        self.predictor_ = PSBPPredictor(traces=traces, burn=burn)

    def _diseno(self, df):
        Xp = np.asarray(df.iloc[:, 1:], dtype=float)
        return np.hstack([np.ones((Xp.shape[0], 1)), Xp])

    def momentos(self, df):
        return self.predictor_.momentos_predictivos(self._diseno(df))

    def muestrear(self, df, d=1, seed=None):
        return self.predictor_.muestrear(self._diseno(df), d, seed=seed)


faltan = [ruta_traza(COMPONENT_IDX[k] + 1, c + 1).name
          for k in range(n_components) for c in range(N_ITER)
          if not ruta_traza(COMPONENT_IDX[k] + 1, c + 1).exists()]
assert not faltan, f"Faltan trazas: {faltan}. Ejecuta psbp_fd_iteracion.m."

models_chains = {k: {} for k in range(n_components)}
for k in range(n_components):
    esperado = list(dfs_train[k].columns[1:])
    for c in range(N_ITER):
        traces, burn, feat = leer_traza(ruta_traza(COMPONENT_IDX[k] + 1, c + 1))
        assert feat == esperado, f"[k={k} c={c+1}] feature_names != columnas del dataset."
        models_chains[k][c] = ModeloTraza(traces, burn, feat)

n_post = MCMC_CFG["nsim"] - BURN
print(f"OK  {n_components} x {N_ITER} cadenas · {n_post} draws posteriores c/u")

## 3. Predicción a $h=1$ sobre la serie completa

Los dos bloques se concatenan en **una sola serie de orígenes**
$t=N_{lags}+1,\dots,T$. Ésa es la diferencia con el flujo anterior, que sólo
evaluaba el bloque de prueba: sin el tramo de entrenamiento no hay con qué
comparar el salto en $T_0$, que es la lectura principal de la ventana móvil.

En todos los orígenes la predicción usa los **rezagos reales** —nunca
predicciones encadenadas—, de modo que el horizonte es 1 en todo el recorrido y
el modelo no se reentrena en ningún punto.

In [ ]:
# Número de extracciones de la predictiva por draw posterior. El total es
#     S = (nsim - burn) × S_POR_ITER × n_cadenas
# y d=1 ya da varios miles: subirlo sólo reduce el error Monte Carlo de estimar
# los cuantiles, no cambia la predictiva.
#
# [FIX memoria] estaba en 100, que con estos tamaños pedía varios GB para
# `SC_draws` en float64. Con 10 el total son decenas de miles de extracciones,
# error MC del cuantil al 2.5% del orden de 0.002 sd, y el cubo cabe holgado.
S_POR_ITER = 10
SEED_PRED  = 20260823

dfs_full = {k: pd.concat([dfs_train[k], dfs_test[k]], ignore_index=True)
            for k in range(n_components)}
n_orig   = len(dfs_full[0])
t_orig   = np.arange(N_LAGS + 1, T + 1)          # tiempo del experimento, base-1
assert len(t_orig) == n_orig, f"{len(t_orig)} != {n_orig}"

T0_orig  = T0 - N_LAGS
es_train = t_orig <= T0

# Estado verdadero alineado a los orígenes evaluados
qb_ev    = quiebre[N_LAGS:]
adapt_ev = en_adapt[N_LAGS:]
assert qb_ev.shape[0] == n_orig

print(f"orígenes evaluados: {n_orig}  (train {es_train.sum()} · test {(~es_train).sum()})")
for j in range(J_ESTRATOS):
    print(f"  {ETIQUETAS[j]:<14} train {int(((qb_ev==j) & es_train).sum()):>4}  "
          f"test {int(((qb_ev==j) & ~es_train).sum()):>4}")
if ESTRATO_ES_PARTICION:
    print("\n  Como se anunció: el estrato coincide con la partición. Los "
          "conteos de arriba\n  lo hacen explícito — cada estrato vive "
          "esencialmente en un solo bloque.")

_mem_gb = n_post * S_POR_ITER * N_ITER * n_orig * n_components * 4 / 1e9
print(f"SC_draws reservará {_mem_gb * 1e3:.0f} MB en float32")
assert _mem_gb < 2.0, (
    f"SC_draws pediría {_mem_gb:.1f} GB. Baja S_POR_ITER antes de continuar.")

In [ ]:
# Momentos y extracciones por score, agrupando cadenas.
#   momentos : ley de varianza total entre cadenas (fit.agrupar_momentos)
#   muestras : concatenación, que es la mezcla de igual peso
#
# [FIX memoria] `SC_draws` se PREASIGNA y cada cadena se escribe en su rebanada.
# El patrón anterior (lista → np.concatenate → np.stack) mantenía vivas tres
# copias del cubo a la vez. float32 basta: estos draws sólo alimentan cuantiles,
# CRPS, PIT y la ventana móvil. La propagación funcional recasta a float64.
Y_obs  = np.column_stack([dfs_full[k].iloc[:, 0].to_numpy() for k in range(n_components)])
Y_hat  = np.empty_like(Y_obs)
Y_sd   = np.empty_like(Y_obs)

S_POR_CADENA = n_post * S_POR_ITER
S_total      = S_POR_CADENA * N_ITER
SC_draws     = np.empty((S_total, n_orig, n_components), dtype=np.float32)

for k in range(n_components):
    medias, sds = [], []
    for j, c in enumerate(sorted(models_chains[k])):
        mom = models_chains[k][c].momentos(dfs_full[k])
        medias.append(mom["media"])
        sds.append(mom["sd"])            # PREDICTIVA (v3), no la del centro
        muestras_c = models_chains[k][c].muestrear(
            dfs_full[k], S_POR_ITER, seed=SEED_PRED + 1000 * k + c)
        assert muestras_c.shape == (S_POR_CADENA, n_orig), (
            f"[k={k} c={c}] muestrear devolvió {muestras_c.shape}; se esperaba "
            f"({S_POR_CADENA}, {n_orig}). ¿nsim/burn del .mat difieren del JSON?")
        SC_draws[j * S_POR_CADENA:(j + 1) * S_POR_CADENA, :, k] = muestras_c
        del muestras_c
    Y_hat[:, k], Y_sd[:, k] = agrupar_momentos(np.column_stack(medias),
                                               np.column_stack(sds))

li_s, ls_s = intervalo_muestral(SC_draws, nivel=NIVEL)   # (n_orig, M)

print(f"extracciones por score: S = {S_total} "
      f"= {n_post} draws × {S_POR_ITER} × {N_ITER} cadenas")
print(f"SC_draws {SC_draws.shape} {SC_draws.dtype} "
      f"({SC_draws.nbytes / 1e6:.0f} MB)  ·  bandas {li_s.shape}")

# Señal temprana del escenario: la varianza REALIZADA de cada score cambia entre
# estratos, mientras que la banda del modelo se calibró con el estrato viejo.
print("\nvarianza realizada del score por estrato (escala estandarizada):")
for k in range(n_components):
    v0 = float(Y_obs[qb_ev == 0, k].var())
    v1 = float(Y_obs[qb_ev == 1, k].var())
    print(f"  FPC {COMPONENT_IDX[k]+1}: {ETIQUETAS[0]} {v0:.4f}   "
          f"{ETIQUETAS[1]} {v1:.4f}   razón {v1 / max(v0, 1e-12):.3f}")
print("   Las componentes intercambiadas deben moverse en direcciones opuestas. "
      "La banda\n   del modelo se calibró con el primer estrato: ahí está el "
      "fallo que §9 mide.")

### 3.1 Propagación a la curva

$\hat X^{(s)}_t(\tau)=\mu(\tau)+\sum_m (d_m\tilde\xi^{(s)}_{tm}+c_m)\psi_m(\tau)$,
que es (3.41) del documento. Con `modo_residuo="ninguno"` el mapa es
determinista y las curvas son extracciones exactas de la predictiva de la curva
**proyectada**.

Las muestras funcionales se **adelgazan** antes de propagar: el arreglo completo
sería $(S, n, G)$ y con $S$ de varios miles ocupa más de un gigabyte sin que los
cuantiles al 95 % mejoren de forma apreciable.

In [ ]:
S_FUNC = 500
paso_thin = max(1, S_total // S_FUNC)
SC_thin = SC_draws[::paso_thin]

propagador = PropagadorFuncional(Psi_grid, mu_grid,
                                 estandarizador=scores_standardizer,
                                 modo_residuo=MODO_RESIDUO)
X_draws = propagador.curvas_desde_scores(SC_thin, seed=SEED_PRED)   # (S', n, G)
X_pred  = X_draws.mean(axis=0)                                       # (n, G)
li_f, ls_f = np.quantile(X_draws, (1 - NIVEL) / 2, axis=0), \
             np.quantile(X_draws, 1 - (1 - NIVEL) / 2, axis=0)

X_true_ev = X_true[N_LAGS:]     # (n_orig, G) VERDADERA — contra esto se evalúa
X_obs_ev  = X_obs[N_LAGS:]      # (n_orig, G) observada, sólo para las figuras

print(f"muestras funcionales {X_draws.shape}  (adelgazado 1 de cada {paso_thin})")
print(f"memoria aprox {X_draws.nbytes / 1e6:.0f} MB")

X_proj = fpca.reconstruct(SCORES)[N_LAGS:]
_mise_trunc = mise(X_true_ev, X_proj, grilla)
_mise_bspl  = float(REPR_CFG.get("mise_bspline_vs_verdadera", np.nan))
print(f"\nMISE del TRUNCAMIENTO (proyección FPCA + B-spline): {_mise_trunc:.6f}")
print(f"  de los cuales, atribuible a la base B-spline    : {_mise_bspl:.6f}")
print("   ← ningún modelo sobre esta representación puede bajar del primero. "
      "En este\n     escenario ese piso NO es constante en el tiempo: §9.1.")

## 4. Métricas, entrenamiento contra prueba

La comparación train/test es la que separa el ajuste de la generalización, con
una salvedad que este escenario obliga a declarar: con $t^{*}=T_0$ el corte
train/test **coincide** con el quiebre de covarianza, de modo que la diferencia
entre las dos columnas no es sólo generalización sino también cambio de
régimen. §9.1 es lo que permite atribuirla.

In [ ]:
def _metricas_bloque(mask, etiqueta):
    filas = []
    for k in range(n_components):
        y, p = Y_obs[mask, k], Y_hat[mask, k]
        z    = SC_draws[:, mask, k]
        cob  = cobertura(y, li_s[mask, k], ls_s[mask, k])
        pit  = pit_muestral(y, z)
        filas.append({
            "bloque":  etiqueta,
            "FPC":     f"FPC {COMPONENT_IDX[k] + 1}",
            "RMSE":    float(rmse_por_coeficiente(y[:, None], p[:, None])[0]),
            "R2":      float(r2_por_columna(y[:, None], p[:, None], centrar=True)[0]),
            "sd_ratio": float(razon_dispersion(y[:, None], p[:, None])[0]),
            "CRPS":    float(crps_muestral(y, z).mean()),
            f"Cob{int(NIVEL*100)}": cob["cobertura"],
            "Ancho":   cob["ancho_medio"],
            "PIT_ks":  float(diagnostico_pit(pit)["ks"]),
            "PIT_forma": diagnostico_pit(pit)["forma"],
            "n":       int(mask.sum()),
        })
    return filas

met_df = pd.DataFrame(_metricas_bloque(es_train, "train")
                      + _metricas_bloque(~es_train, "test"))
met_df = met_df.set_index(["bloque", "FPC"])
met_df.to_csv(PATHS["out_report"] / "50_metricas_scores.csv")

_num = [c for c in met_df.columns if met_df[c].dtype.kind == "f"]
display(met_df.style
    .format({c: "{:.4f}" for c in _num})
    .background_gradient(subset=["RMSE", "CRPS"], cmap="RdYlGn_r")
    .background_gradient(subset=["R2"], cmap="RdYlGn", vmin=0, vmax=1)
    .background_gradient(subset=[f"Cob{int(NIVEL*100)}"], cmap="RdYlGn",
                         vmin=0.80, vmax=1.0)
    .set_caption("Métricas por score y bloque · CRPS y cobertura desde la "
                 "predictiva MUESTRAL"))

_deg_rmse = met_df.loc["test", "RMSE"].mean() / met_df.loc["train", "RMSE"].mean()
_cob_tr   = met_df.loc["train", f"Cob{int(NIVEL*100)}"].mean()
_cob_te   = met_df.loc["test",  f"Cob{int(NIVEL*100)}"].mean()
print(f"\nRMSE test / RMSE train      = {_deg_rmse:.3f}")
print(f"cobertura media  train={_cob_tr:.4f}  test={_cob_te:.4f}  "
      f"caída {_cob_tr - _cob_te:+.4f}")
print("\nLectura: si la cobertura cae mucho más de lo que sube el RMSE, el "
      "fallo es de\nCALIBRACIÓN —la banda se calibró con la varianza del "
      "régimen viejo— y no del\ncentro. Es la firma esperable de este "
      "escenario.")
if ESTRATO_ES_PARTICION:
    print("\n[RECORDAR] Con t* = T0 esta comparación NO separa generalización "
          "de vigencia\n  de la base. §9.1 es la que lo hace.")

In [ ]:
# Métricas funcionales, contra la curva VERDADERA
filas_f = []
for mask, etq in ((es_train, "train"), (~es_train, "test")):
    dentro = (X_true_ev[mask] >= li_f[mask]) & (X_true_ev[mask] <= ls_f[mask])
    filas_f.append({
        "bloque": etq,
        "MISE":   mise(X_true_ev[mask], X_pred[mask], grilla),
        "RMSE_f": rmse_funcional(X_true_ev[mask], X_pred[mask], grilla),
        "MISE_truncamiento": mise(X_true_ev[mask], X_proj[mask], grilla),
        "energy": energy_score(X_true_ev[mask], X_draws[:, mask, :],
                               max_pares=2000, seed=0),
        f"Cob{int(NIVEL*100)}_puntual": float(dentro.mean()),
        "ancho_medio": float((ls_f[mask] - li_f[mask]).mean()),
        "n": int(mask.sum()),
    })
fun_df = pd.DataFrame(filas_f).set_index("bloque")
fun_df.to_csv(PATHS["out_report"] / "51_metricas_funcionales.csv")

display(fun_df.style.format("{:.6f}", subset=["MISE", "RMSE_f", "MISE_truncamiento"])
        .format("{:.4f}", subset=["energy", f"Cob{int(NIVEL*100)}_puntual", "ancho_medio"])
        .set_caption("Métricas funcionales contra la curva VERDADERA · "
                     "banda sin residuo de representación"))

_rt = float(fun_df.loc["test", "MISE_truncamiento"] /
            fun_df.loc["train", "MISE_truncamiento"])
print(f"\nMISE_truncamiento  test / train = {_rt:.3f}")
print("   ESTA es la cifra del escenario, y no depende del modelo predictivo: "
      "es la\n   distancia entre la curva verdadera y el subespacio de las M "
      "autofunciones\n   ajustadas en entrenamiento. Si crece, la base perdió "
      "vigencia. §9.1 la\n   desglosa en el tiempo.")

## 5. Bandas de credibilidad sobre la serie de scores

In [ ]:
plot_bandas_serie(
    Y_obs, Y_hat, li_s, ls_s, T0, t=t_orig,
    etiquetas=[f"FPC {i + 1}" for i in COMPONENT_IDX], nivel=NIVEL,
    title="Bandas de credibilidad por score — entrenamiento y prueba",
    save_path=str(PATHS["out_report"] / "52_bandas_scores.png"))
plt.show()
print("Lectura del escenario: la banda tiene ANCHO CONSTANTE —se calibró con el "
      "régimen\nviejo— mientras que la dispersión realizada de los scores cambia "
      "en t*. En la\ncomponente que GANA varianza la serie debe empezar a "
      "salirse de la banda; en la\nque la PIERDE, la banda queda "
      "innecesariamente ancha.")

# Cuantificación: ancho de banda contra dispersión realizada, por estrato.
print("\nancho medio de la banda vs sd realizada del score, por estrato:")
for k in range(n_components):
    for j in range(J_ESTRATOS):
        m = qb_ev == j
        if m.sum() == 0:
            continue
        _anc = float((ls_s[m, k] - li_s[m, k]).mean())
        _sd  = float(Y_obs[m, k].std())
        print(f"  FPC {COMPONENT_IDX[k]+1} · {ETIQUETAS[j]:<14} "
              f"ancho {_anc:.4f}   sd realizada {_sd:.4f}   "
              f"razón {_anc / max(_sd, 1e-12):.3f}")
print("   Bajo calibración correcta la razón sería aproximadamente constante "
      "entre\n   estratos. Que no lo sea es la medición.")

In [ ]:
eval_scatter = {k: {"y_obs": Y_obs[~es_train, k], "y_hat": Y_hat[~es_train, k]}
                for k in range(n_components)}
plot_scatter_theta(eval_scatter, n_components=n_components,
                   save_path=str(PATHS["out_report"] / "53_scatter_scores_test.png"))
plt.show()

## 6. Intervalos de credibilidad sobre la curva

Extractos de la serie funcional cada `CADA` períodos, cada uno con su banda
puntual, la curva verdadera y —en gris— los datos observados, para ver de un
vistazo dónde estaba el ruido que la banda **no** tiene que cubrir.

In [ ]:
CADA = 10   # [CONFIG] un extracto cada CADA períodos

plot_extractos_curvas(
    X_true_ev, X_pred, li_f, ls_f, grilla, T0, t=t_orig,
    cada=CADA, n_col=5, nivel=NIVEL, X_obs=X_obs_ev,
    title="Predictiva funcional y banda de credibilidad",
    save_path=str(PATHS["out_report"] / "54_extractos_curvas.png"))
plt.show()

In [ ]:
# Zoom sobre la frontera train/test, que aquí es TAMBIÉN el quiebre.
i_corte = int(np.searchsorted(t_orig, T0))
sel = np.arange(max(0, i_corte - 4), min(n_orig, i_corte + 5))

plot_extractos_curvas(
    X_true_ev[sel], X_pred[sel], li_f[sel], ls_f[sel], grilla, T0,
    t=t_orig[sel], cada=1, n_col=len(sel), nivel=NIVEL, X_obs=X_obs_ev[sel],
    title=f"Frontera train/test = quiebre del espectro (T0 = t* = {T0})",
    save_path=str(PATHS["out_report"] / "55_extractos_frontera.png"))
plt.show()
print("A diferencia del Algoritmo 5 —donde la frontera no muestra nada— aquí "
      "el cambio\nde régimen de covarianza ocurre exactamente en este punto. "
      "La forma de las curvas\ndebe cambiar, no sólo su escala: el "
      "reordenamiento afecta a QUÉ dirección domina.")

## 7. Ventana móvil

El modelo **no se reentrena**: lo que se desliza es la ventana de evaluación.
Cada punto agrega las métricas de $w$ orígenes consecutivos, todos a $h=1$ con
los rezagos reales. Se superponen varios anchos para que la conclusión no
dependa de un $w$ elegido a dedo; las ventanas que **cruzan** $T_0$ van
punteadas, porque su cifra mezcla dentro y fuera de muestra.

In [ ]:
tablas_score = {
    w: ventana_movil_scores(
        Y_obs, Y_hat, T0_orig, w=w, muestras=SC_draws, li=li_s, ls=ls_s,
        t_offset=N_LAGS,
        etiquetas=[f"FPC {i + 1}" for i in COMPONENT_IDX])
    for w in VENTANAS_W
}
w_ref = VENTANAS_W[len(VENTANAS_W) // 2]
tablas_score[w_ref].to_csv(PATHS["out_report"] / f"57_ventana_scores_w{w_ref}.csv",
                           index=False)

plot_ventana_movil(
    tablas_score[w_ref], T0, ["rmse", "r2_local", "crps", "cobertura"],
    columna_grupo="componente",
    title=f"Ventana móvil por score (w={w_ref})",
    save_path=str(PATHS["out_report"] / "57_ventana_scores.png"))
plt.show()

In [ ]:
tablas_fun = {
    w: ventana_movil_funcional(X_true_ev, X_pred, grilla, T0_orig, w=w,
                               li=li_f, ls=ls_f, t_offset=N_LAGS)
    for w in VENTANAS_W
}
tablas_fun[w_ref].to_csv(PATHS["out_report"] / f"58_ventana_funcional_w{w_ref}.csv",
                         index=False)

plot_ventana_movil(
    tablas_fun[w_ref], T0, ["mise", "mise_rel", "cobertura_puntual"],
    tablas_por_w=tablas_fun,
    title="Ventana móvil del error funcional (contra la curva verdadera)",
    save_path=str(PATHS["out_report"] / "58_ventana_funcional.png"))
plt.show()

In [ ]:
# El MISE del MODELO y el del TRUNCAMIENTO en la misma ventana móvil. Si suben
# juntos, la degradación es de la representación y no del ajuste.
t_ref  = tablas_fun[w_ref]
centro = t_ref["t_fin"].to_numpy() if "t_fin" in t_ref.columns else None

if centro is not None:
    _err_trunc = (((X_true_ev - X_proj) ** 2) @ w_quad)      # (n_orig,) por origen
    mise_tr_v = np.array([
        float(np.mean(_err_trunc[max(0, int(tf) - N_LAGS - w_ref):
                                 max(1, int(tf) - N_LAGS)]))
        for tf in centro])

    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot(centro, t_ref["mise"], lw=1.4, color="#2c3e50",
            label="MISE del MODELO")
    ax.plot(centro, mise_tr_v, lw=1.4, color="#e67e22", ls="--",
            label="MISE del TRUNCAMIENTO (no depende del modelo)")
    ax.axvline(T0, color="k", ls="--", lw=1.2)
    ax.text(T0, ax.get_ylim()[1], r"  $T_0=t^*$", fontsize=9, va="top")
    if N_ADAPT > 0:
        ax.axvspan(T0, T0 + N_ADAPT, color="0.75", alpha=0.4)
    ax.set_xlabel("$t$ final de la ventana"); ax.set_ylabel("MISE local")
    ax.legend(fontsize=8)
    ax.set_title(f"Error del modelo vs error de la representación (w={w_ref}) — "
                 "si suben juntos, la base es la que falla", fontsize=11)
    fig.tight_layout()
    fig.savefig(PATHS["out_report"] / "59_mise_modelo_vs_truncamiento.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    _c = float(np.corrcoef(t_ref["mise"].to_numpy(), mise_tr_v)[0, 1])
    print(f"corr(MISE del modelo, MISE del truncamiento) = {_c:+.3f}")
    print("   Alta y positiva: la degradación del modelo SIGUE a la de la "
          "representación,\n   que es la tesis del escenario.")

In [ ]:
# Lectura numérica del salto en T0, excluyendo las ventanas que lo cruzan.
print(f"{'w':>4}  {'MISE train':>11}  {'MISE test':>11}  {'salto':>7}")
print(f"{'-'*4}  {'-'*11}  {'-'*11}  {'-'*7}")
for w in VENTANAS_W:
    t_ = tablas_fun[w]
    limpio = t_[~t_["cruza_T0"]]
    a = limpio.loc[limpio.bloque == "train", "mise"].mean()
    b = limpio.loc[limpio.bloque == "test",  "mise"].mean()
    print(f"{w:>4}  {a:>11.6f}  {b:>11.6f}  {b/a:>6.2f}x")

print("\nEn la corrida 15 —mismo Bloque 2, proceso estacionario— este salto "
      "debería ser\ncercano a 1. Aquí no. La diferencia entre ambos es el "
      "efecto del quiebre.")

## 8. Calibración marginal

Bajo calibración perfecta el PIT es uniforme. La **forma** dice qué falla: una
U indica bandas demasiado angostas, una campana demasiado anchas, y una
pendiente un sesgo del centro. Contrastar train contra test separa el desajuste
del modelo de la pérdida de calibración fuera de muestra.

In [ ]:
pit_tr = {f"FPC {COMPONENT_IDX[k]+1}": pit_muestral(Y_obs[es_train, k],
                                                    SC_draws[:, es_train, k])
          for k in range(n_components)}
pit_te = {f"FPC {COMPONENT_IDX[k]+1}": pit_muestral(Y_obs[~es_train, k],
                                                    SC_draws[:, ~es_train, k])
          for k in range(n_components)}

plot_calibracion_pit(pit_tr, pit_te,
                     save_path=str(PATHS["out_report"] / "60_pit.png"))
plt.show()

for nombre in pit_tr:
    d_tr, d_te = diagnostico_pit(pit_tr[nombre]), diagnostico_pit(pit_te[nombre])
    print(f"  {nombre}:  train KS={d_tr['ks']:.3f} ({d_tr['forma']})   "
          f"test KS={d_te['ks']:.3f} ({d_te['forma']})")

_formas_te = {n: diagnostico_pit(p)["forma"] for n, p in pit_te.items()}
print(f"\nformas del PIT en prueba: {_formas_te}")
if len(set(_formas_te.values())) > 1:
    print("   Formas DISTINTAS entre componentes: es la firma del "
          "reordenamiento. Unas\n   bandas quedaron cortas y otras largas, y "
          "una lectura agregada las cancelaría.")

## 9. Calibración condicional al régimen de covarianza

Eje 2 del diseño (`docs/03 Modelo.tex §03_06`). El estado verdadero es aquí
**discreto** —pre o post quiebre— y se pasa directo a `cobertura_condicional`,
sin cuantilizar, exactamente como el régimen en la corrida 13.

**Y aquí va, por tercera vez y en el lugar donde se leen los números:**

> Con $t^{*}=T_0$ este estrato **coincide con la partición train/test**. Esta
> sección y la comparación train/test de §4 miden lo mismo. La cobertura que
> caiga en el estrato post-quiebre **no** es evidencia de mala generalización:
> es evidencia de que la base perdió vigencia. §9.1 es la que lo demuestra.

Qué esperar:

- **Cobertura cercana a la nominal en pre-quiebre y por debajo en post**, con
  anchos parecidos: la banda se calibró con la varianza vieja y no se adapta.
- **Anchos parecidos entre estratos** es la observación clave. Si la banda se
  hubiera adaptado, el ancho habría cambiado; que no cambie muestra que el
  modelo no puede ver lo que no está en sus coordenadas.

Se reporta además la versión que **excluye los períodos de adaptación**, en los
que la varianza marginal todavía converge al nuevo nivel y que por tanto no
pertenecen limpiamente a ninguno de los dos regímenes.

In [ ]:
filas_cc = []
for mask, etq_bloque in ((es_train, "train"), (~es_train, "test"),
                         (np.ones(n_orig, bool), "todo"),
                         (~adapt_ev, "sin adaptación")):
    presentes = np.unique(qb_ev[mask])
    if presentes.size == 0:
        continue
    fs = cobertura_condicional(
        X_true_ev[mask], li_f[mask], ls_f[mask], qb_ev[mask],
        etiquetas=[ETIQUETAS[j] for j in presentes],
        tau=grilla, muestras=X_draws[:, mask, :])
    for f in fs:
        filas_cc.append({"bloque": etq_bloque, **f})

cc_df = pd.DataFrame(filas_cc).set_index(["bloque", "estrato"])
cc_df.to_csv(PATHS["out_report"] / "61_cobertura_condicional_funcional.csv")

display(cc_df.style
    .format({"cobertura": "{:.4f}", "ancho_medio": "{:.4f}",
             "desvio": "{:+.4f}", "energy": "{:.4f}"})
    .background_gradient(subset=["cobertura"], cmap="RdYlGn", vmin=0.80, vmax=1.0)
    .set_caption(f"Cobertura funcional al {NIVEL:.0%} por RÉGIMEN DE COVARIANZA · "
                 "`desvio` = estrato − marginal"))

_todo = cc_df.loc["todo"]
print(f"cobertura marginal (test)   : "
      f"{fun_df.loc['test', f'Cob{int(NIVEL*100)}_puntual']:.4f}")
print(f"rango entre estratos (todo) : "
      f"[{_todo['cobertura'].min():.4f}, {_todo['cobertura'].max():.4f}]   "
      f"amplitud {_todo['cobertura'].max() - _todo['cobertura'].min():.4f}")
print(f"ancho medio por estrato     : "
      f"{[round(float(a), 4) for a in _todo['ancho_medio']]}")

_amp = float(_todo["cobertura"].max() - _todo["cobertura"].min())
_anc = _todo["ancho_medio"].to_numpy()
_var_anc = float(abs(_anc.max() - _anc.min()) / max(_anc.mean(), 1e-12))
print(f"\nvariación relativa del ancho entre estratos: {_var_anc:.1%}")
print("VEREDICTO del eje 2: "
      + ("cobertura estable entre regímenes de covarianza" if _amp < 0.05 else
         "la cobertura DEPENDE del régimen de covarianza"))
if _amp >= 0.05 and _var_anc < 0.10:
    print("   Y lo hace SIN que el ancho cambie: la banda no se adapta porque "
          "el cambio\n   ocurre en direcciones que la representación ya no "
          "ordena. Es la firma exacta\n   de la obsolescencia de la base.")
if ESTRATO_ES_PARTICION:
    print("\n[RECORDAR] estrato == partición. Esta sección NO prueba por sí "
          "sola que el\n  fallo sea de la base y no de generalización. §9.1 es "
          "la prueba.")

In [ ]:
# Cobertura condicional por SCORE. Aquí se ve el efecto OPUESTO entre las dos
# componentes intercambiadas, que la versión funcional agrega y esconde.
filas_cs = []
for k in range(n_components):
    presentes = np.unique(qb_ev)
    fs = cobertura_condicional(
        Y_obs[:, k], li_s[:, k], ls_s[:, k], qb_ev,
        etiquetas=[ETIQUETAS[j] for j in presentes],
        muestras=SC_draws[:, :, k])
    for f in fs:
        filas_cs.append({"FPC": f"FPC {COMPONENT_IDX[k] + 1}", **f})

cs_df = pd.DataFrame(filas_cs).set_index(["FPC", "estrato"])
cs_df.to_csv(PATHS["out_report"] / "62_cobertura_condicional_scores.csv")

display(cs_df.style
    .format({"cobertura": "{:.4f}", "ancho_medio": "{:.4f}",
             "desvio": "{:+.4f}", "crps": "{:.4f}"})
    .background_gradient(subset=["cobertura"], cmap="RdYlGn", vmin=0.80, vmax=1.0)
    .set_caption("Cobertura por score y régimen de covarianza"))

_fpc_int = REPR_CFG.get("fpc_intercambiadas", [])
print(f"componentes FPCA alineadas con las intercambiadas: {_fpc_int}")
print("En ellas la cobertura debe moverse en direcciones OPUESTAS entre "
      "estratos: la que\ngana varianza queda sub-cubierta, la que la pierde "
      "sobre-cubierta. La versión\nfuncional de la tabla anterior las agrega y "
      "puede cancelarlas.")

## 9.1 Obsolescencia de la base

**El resultado central de esta corrida**, y la única sección que puede
distinguir pérdida de vigencia de la base de pérdida de generalización.

El argumento es el siguiente. El **error de representación** en el origen $t$,

$$e_t=\bigl\|X_t-\widehat{\Pi}_M X_t\bigr\|^2_{L^2},$$

con $\widehat{\Pi}_M$ la proyección sobre las $M$ autofunciones **ajustadas en
entrenamiento**, no depende del modelo predictivo en absoluto: es la distancia
entre la curva verdadera y el subespacio. Si $e_t$ crece tras $t^{*}$, ese
crecimiento es atribuible **únicamente** a que el subespacio dejó de ser el
adecuado.

Se compara contra dos referencias:

1. el $e_t$ medio del **bloque de entrenamiento**, que es el nivel que la base
   alcanza en el régimen para el que se ajustó;
2. el error de una base **reajustada sobre el bloque de prueba** —que este
   notebook calcula sólo como cota inferior de referencia, sin usarla para
   predecir nada—. La brecha entre ambos es lo que se ganaría reajustando la
   representación, y es la magnitud que este escenario existe para cuantificar.

La segunda referencia es una **cota optimista** —usa el test para ajustarse a sí
mismo— y por eso no puede leerse como el desempeño de un método alternativo.
Sirve para acotar el tamaño del problema, no para comparar modelos.

In [ ]:
# ── e_t: error de representación por origen, con la base de ENTRENAMIENTO ────
err_repr = ((X_true_ev - X_proj) ** 2) @ w_quad          # (n_orig,)

e_tr   = float(err_repr[es_train].mean())
e_te   = float(err_repr[~es_train].mean())
e_pre  = float(err_repr[qb_ev == 0].mean())
e_post = float(err_repr[qb_ev == 1].mean())
e_post_limpio = (float(err_repr[(qb_ev == 1) & ~adapt_ev].mean())
                 if ((qb_ev == 1) & ~adapt_ev).sum() > 0 else np.nan)

print("Error de representación medio (NO depende del modelo predictivo):")
print(f"  entrenamiento               : {e_tr:.6f}")
print(f"  prueba                      : {e_te:.6f}   razón {e_te/e_tr:.3f}x")
print(f"  {ETIQUETAS[0]:<26}: {e_pre:.6f}")
print(f"  {ETIQUETAS[1]:<26}: {e_post:.6f}   razón {e_post/e_pre:.3f}x")
print(f"  {ETIQUETAS[1]} sin adaptación : {e_post_limpio:.6f}")

In [ ]:
# ── Cota de referencia: una base reajustada SOBRE EL BLOQUE DE PRUEBA ───────
# Sólo como cota inferior. No se usa para predecir nada, y por construcción es
# optimista: se ajusta a los mismos datos sobre los que se evalúa.
from model_psbp_fd.functions_models import FPCA_L2, base_en_grilla
from model_psbp_fd.pipelines import cargar_representacion

fr_cargada, THETA_all, _fr_meta = cargar_representacion(PATHS)
Phi_grid_base = base_en_grilla(fr_cargada, THETA_all.shape[1])
idx_test_full = np.arange(T0, T)

fpca_te = FPCA_L2().fit(THETA_all[idx_test_full], Phi_grid_base, grilla)
fpca_te.set_M(int(M_fpca))
X_proj_te = fpca_te.reconstruct(fpca_te.transform(THETA_all[idx_test_full]))
err_repr_reajustada = ((X_true[idx_test_full] - X_proj_te) ** 2) @ w_quad
e_te_reajustada = float(err_repr_reajustada.mean())

print(f"error de representación en el bloque de PRUEBA:")
print(f"  con la base de entrenamiento (la que se usa) : {e_te:.6f}")
print(f"  con una base reajustada al test (COTA)       : {e_te_reajustada:.6f}")
print(f"  brecha atribuible a la obsolescencia         : "
      f"{e_te - e_te_reajustada:.6f}   "
      f"({(e_te - e_te_reajustada) / max(e_te, 1e-15):.1%} del error total)")
print("\n  La cota es OPTIMISTA por construcción: se ajusta a los mismos datos "
      "sobre los\n  que se evalúa. Acota el tamaño del problema; no es el "
      "desempeño de un método\n  alternativo y no debe reportarse como tal.")

In [ ]:
# ── Figura: el error de representación a lo largo del tiempo ────────────────
_w = 20
_suav = np.array([float(err_repr[max(0, i - _w):i + 1].mean())
                  for i in range(n_orig)])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 3.8),
                         gridspec_kw={"width_ratios": [3, 1]})

ax = axes[0]
ax.plot(t_orig, err_repr, lw=0.6, color="0.7", label="$e_t$ por origen")
ax.plot(t_orig, _suav, lw=1.8, color="#2c3e50",
        label=f"media móvil (w={_w})")
ax.axhline(e_tr, color="#27ae60", ls=":", lw=1.6,
           label="nivel en entrenamiento")
ax.axhline(e_te_reajustada, color="#e67e22", ls="-.", lw=1.6,
           label="cota: base reajustada al test")
ax.axvline(T0, color="k", ls="--", lw=1.4)
ax.text(T0, ax.get_ylim()[1], r"  $T_0=t^*$", fontsize=9, va="top")
if N_ADAPT > 0:
    ax.axvspan(T0, T0 + N_ADAPT, color="0.8", alpha=0.5)
ax.set_xlabel("$t$"); ax.set_ylabel(r"$\|X_t-\widehat{\Pi}_M X_t\|^2$")
ax.legend(fontsize=8)
ax.set_title("Error de representación con la base ajustada en entrenamiento",
             fontsize=10)

ax = axes[1]
ax.hist(err_repr[qb_ev == 0], bins=35, alpha=0.6, color="#2980b9",
        density=True, label=ETIQUETAS[0])
ax.hist(err_repr[qb_ev == 1], bins=35, alpha=0.6, color="#c0392b",
        density=True, label=ETIQUETAS[1])
ax.legend(fontsize=8)
ax.set_title("distribución de $e_t$ por régimen", fontsize=9)

fig.suptitle("Escenario 6 — la base pierde vigencia en $t^*$: el error crece "
             "sin que el modelo cambie", fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "63_obsolescencia_base.png", dpi=150,
            bbox_inches="tight")
plt.show()

obsol_df = pd.DataFrame({
    "t": t_orig, "error_representacion": err_repr,
    "quiebre_idx": qb_ev, "en_adaptacion": adapt_ev,
    "bloque": np.where(es_train, "train", "test"),
})
obsol_df.to_csv(PATHS["out_report"] / "63_obsolescencia_base.csv", index=False)
print(f"[out_report] 63_obsolescencia_base.csv  {obsol_df.shape}")

In [ ]:
# ── Veredicto de §9.1, y de la corrida ─────────────────────────────────────
RAZON_OBSOL = float(e_post / max(e_pre, 1e-15))
FRAC_BRECHA = float((e_te - e_te_reajustada) / max(e_te, 1e-15))

print("=" * 72)
print("VEREDICTO — obsolescencia de la base")
print(f"""
  El error de representación pasa de {e_pre:.6f} en {ETIQUETAS[0]} a
  {e_post:.6f} en {ETIQUETAS[1]}: un factor de {RAZON_OBSOL:.2f}x.

  Esa cantidad NO depende del modelo predictivo: es la distancia entre la
  curva verdadera y el subespacio de las {M_fpca} autofunciones ajustadas en
  entrenamiento. Su crecimiento es atribuible únicamente a que el subespacio
  dejó de ser el adecuado.

  De ahí se sigue la conclusión que separa este escenario de una simple
  pérdida de generalización: el {FRAC_BRECHA:.1%} del error de representación del
  bloque de prueba desaparecería con sólo reajustar la base, sin tocar el
  modelo predictivo.""")

if RAZON_OBSOL > 1.2:
    print("""
  La base ajustada en entrenamiento ha perdido vigencia. La degradación de
  §4 y la caída de cobertura de §9 tienen aquí su explicación, y NO son
  atribuibles al PSBPM-FD: alcanzarían por igual a cualquier método sobre
  esta misma representación.""")
else:
    print("""
  El error de representación apenas crece: el reordenamiento del espectro no
  se tradujo en pérdida de vigencia apreciable de la base. Con R = 1 y M
  moderado puede ocurrir si las autofunciones de ambos regímenes generan un
  subespacio similar aunque su ORDEN cambie —la proyección no depende del
  orden—. En ese caso el escenario mide el efecto sobre la CALIBRACIÓN
  (§9, donde la varianza por dirección sí cambia) y no sobre el error de
  representación. Reportarlo así.""")
print("=" * 72)

## 10. Comparación con las líneas base

In [ ]:
_bl = PATHS["out_report"] / "30_baselines_test.csv"
rmse_psbp = met_df.loc["test", "RMSE"].mean()
print(f"RMSE promedio en scores — PSBP-FD (test): {rmse_psbp:.4f}\n")

if _bl.exists():
    baselines_df = pd.read_csv(_bl, index_col=0)
    display(baselines_df.style.format("{:.4f}", na_rep="—")
            .set_caption("Líneas base sobre el bloque de prueba (h=1)"))
    if "RMSE_scores_prom" in baselines_df.columns:
        for modelo, fila in baselines_df.iterrows():
            marca = "PSBP mejor" if rmse_psbp < fila["RMSE_scores_prom"] else "baseline mejor"
            print(f"  vs {modelo:24s} RMSE={fila['RMSE_scores_prom']:.4f}  → {marca}")
else:
    print("[AVISO] falta 30_baselines_test.csv — ejecuta 16_01 §6.")

print("\nLectura para el Escenario 6: las líneas base operan sobre los MISMOS "
      "scores y\nsufren la obsolescencia de la base exactamente igual. Que el "
      "PSBPM-FD y las\nlíneas base se degraden EN LA MISMA PROPORCIÓN al cruzar "
      "t* es el resultado\nesperado, y es la forma de mostrar que la "
      "degradación es de la representación y\nno del modelo.")
print("\nNota: sólo hay dos líneas base (media incondicional y persistencia). "
      "FAR(1),\nVAR sobre scores y ARIMA por score siguen pendientes en "
      "fit/baselines.py.")

In [ ]:
# ── Resumen ejecutable del experimento ───────────────────────────────────────
_todo = cc_df.loc["todo"]
resumen = {
    "experiment_id": EXPERIMENT_ID,
    "escenario_id": int(ESCENARIO_ID),
    "bloque_del_anexo": 2,
    "objetivo_evaluacion": OBJETIVO,
    "modo_residuo": MODO_RESIDUO,
    "nivel": NIVEL,
    "S_scores": int(S_total), "S_funcional": int(X_draws.shape[0]),
    # Diseño
    "t_quiebre": int(TQ), "T0": int(T0),
    "estrato_coincide_con_particion": bool(ESTRATO_ES_PARTICION),
    "M_retenidas": int(M_fpca),
    "fpc_intercambiadas": REPR_CFG.get("fpc_intercambiadas", []),
    # Puntual
    "rmse_scores_train": float(met_df.loc["train", "RMSE"].mean()),
    "rmse_scores_test":  float(met_df.loc["test", "RMSE"].mean()),
    "r2_scores_test":    float(met_df.loc["test", "R2"].mean()),
    "crps_scores_test":  float(met_df.loc["test", "CRPS"].mean()),
    # Funcional
    "mise_train": float(fun_df.loc["train", "MISE"]),
    "mise_test":  float(fun_df.loc["test", "MISE"]),
    "mise_truncamiento_train": float(fun_df.loc["train", "MISE_truncamiento"]),
    "mise_truncamiento_test":  float(fun_df.loc["test", "MISE_truncamiento"]),
    "energy_test": float(fun_df.loc["test", "energy"]),
    "cobertura_puntual_test": float(fun_df.loc["test", f"Cob{int(NIVEL*100)}_puntual"]),
    # Eje 2
    "cobertura_pre":  float(_todo["cobertura"].iloc[0]),
    "cobertura_post": float(_todo["cobertura"].iloc[-1]),
    "amplitud_cobertura_regimenes": float(_todo["cobertura"].max()
                                          - _todo["cobertura"].min()),
    "ancho_pre":  float(_todo["ancho_medio"].iloc[0]),
    "ancho_post": float(_todo["ancho_medio"].iloc[-1]),
    # §9.1 — el resultado central
    "error_repr_pre":  float(e_pre),
    "error_repr_post": float(e_post),
    "razon_obsolescencia": float(RAZON_OBSOL),
    "error_repr_test_base_reajustada": float(e_te_reajustada),
    "fraccion_error_atribuible_a_obsolescencia": float(FRAC_BRECHA),
}
pd.Series(resumen).to_csv(PATHS["out_report"] / "65_resumen.csv", header=False)
for k, v in resumen.items():
    print(f"  {k:44s}: {v}")
print(f"\nFiguras y tablas en {PATHS['out_report']}")
print("""
Al reportar, encabezar SIEMPRE con las dos advertencias:
  1. Bloque 2: el resultado no discrimina entre especificaciones dinámicas;
     acota el alcance de la reducción de dimensión.
  2. t* = T0: el estrato coincide con la partición train/test, de modo que la
     caída de cobertura es pérdida de VIGENCIA DE LA BASE (§9.1 lo demuestra)
     y no pérdida de generalización.""")